# Risk-neutral pricing by simulation --- companion to *Valuation of Options, part 3*

This notebook makes the change of measure $\mathbb{P}\to\mathbb{Q}$ concrete:
1. simulate $S_T$ under the real-world $\mathbb{P}$ (drift $\mu$) and the risk-neutral $\mathbb{Q}$ (drift $r$) and compare the two laws;
2. look at sample paths under each measure;
3. price a European call by **Monte Carlo under $\mathbb{Q}$** and check it against the Black-Scholes formula --- and see that discounting under $\mathbb{P}$ gives the *wrong* price.

Only `numpy` and `matplotlib` are required.


## 0. Imports and market


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, log, sqrt, exp
rng = np.random.default_rng(0)

def Phi(x):                       # standard normal CDF, no SciPy
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

S0, mu, r, sigma, T = 100.0, 0.12, 0.03, 0.20, 1.0
gamma = (mu - r) / sigma          # market price of risk
print(f'market price of risk gamma = (mu-r)/sigma = {gamma:.3f}')


## 1. The law of $S_T$ under $\mathbb{P}$ and under $\mathbb{Q}$

Under a measure with drift $d$, $\;S_T=S_0\exp\big((d-\tfrac12\sigma^2)T+\sigma\sqrt{T}\,Z\big)$, $Z\sim N(0,1)$. Use $d=\mu$ for $\mathbb{P}$ and $d=r$ for $\mathbb{Q}$.


In [ ]:
def terminal(drift, n):
    Z = rng.standard_normal(n)
    return S0 * np.exp((drift - 0.5*sigma**2)*T + sigma*sqrt(T)*Z)

n = 200_000
ST_P = terminal(mu, n)
ST_Q = terminal(r,  n)
print(f'E_P[S_T]: simulated {ST_P.mean():.2f}  vs theory {S0*exp(mu*T):.2f}')
print(f'E_Q[S_T]: simulated {ST_Q.mean():.2f}  vs theory {S0*exp(r*T):.2f}')


In [ ]:
grid = np.linspace(40, 200, 400)
def lognormal_pdf(s, drift):
    m = log(S0) + (drift - 0.5*sigma**2)*T; sd = sigma*sqrt(T)
    return np.exp(-(np.log(s)-m)**2/(2*sd**2)) / (s*sd*sqrt(2*np.pi))

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(ST_P, bins=120, range=(40,200), density=True, alpha=0.3, color='b')
ax.hist(ST_Q, bins=120, range=(40,200), density=True, alpha=0.3, color='r')
ax.plot(grid, lognormal_pdf(grid, mu), 'b', label=r'$\mathbb{P}$ (drift $\mu$)')
ax.plot(grid, lognormal_pdf(grid, r),  'r', label=r'$\mathbb{Q}$ (drift $r$)')
ax.set_xlabel('$S_T$'); ax.set_ylabel('density'); ax.legend(); plt.show()


## 2. Sample paths under $\mathbb{P}$ and $\mathbb{Q}$

The two measures share the *same* Brownian increments (Girsanov only changes the drift); $\mathbb{P}$-paths simply trend upward faster, by the risk premium $\mu-r$.


In [ ]:
steps, paths = 250, 12
dt = T/steps; t = np.linspace(0, T, steps+1)
dW = rng.standard_normal((paths, steps)) * sqrt(dt)   # shared increments
def make_paths(drift):
    incr = (drift - 0.5*sigma**2)*dt + sigma*dW
    logS = np.concatenate([np.zeros((paths,1)), np.cumsum(incr, axis=1)], axis=1)
    return S0*np.exp(logS)
SP, SQ = make_paths(mu), make_paths(r)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(t, SP.T, 'b', alpha=0.4, lw=0.8)
ax.plot(t, SQ.T, 'r', alpha=0.4, lw=0.8)
ax.plot([], [], 'b', label=r'$\mathbb{P}$ (drift $\mu$)'); ax.plot([], [], 'r', label=r'$\mathbb{Q}$ (drift $r$)')
ax.set_xlabel('t'); ax.set_ylabel('$S_t$'); ax.legend(); plt.show()


## 3. Monte-Carlo price vs Black-Scholes

The no-arbitrage price is the **discounted $\mathbb{Q}$-expectation** of the payoff: $C_0=e^{-rT}\,\mathbb{E}_{\mathbb{Q}}[(S_T-K)^+]$. Discounting the *real-world* expectation instead gives the wrong number.


In [ ]:
def bs_call(S0, K, r, sigma, T):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T)/(sigma*sqrt(T)); d2 = d1 - sigma*sqrt(T)
    return S0*Phi(d1) - K*exp(-r*T)*Phi(d2)

K = 100.0
mc_Q = exp(-r*T) * np.maximum(ST_Q - K, 0).mean()   # correct: discount E_Q
mc_P = exp(-r*T) * np.maximum(ST_P - K, 0).mean()   # wrong: discount E_P
exact = bs_call(S0, K, r, sigma, T)
print(f'Black-Scholes (exact)            : {exact:.4f}')
print(f'Monte-Carlo under Q (correct)    : {mc_Q:.4f}')
print(f'Monte-Carlo under P (WRONG)      : {mc_P:.4f}   <-- overprices by the risk premium')


**Takeaway.** Simulating under $\mathbb{Q}$ and discounting at $r$ reproduces the Black-Scholes price; using the real-world drift $\mu$ does not. The drift $\mu$ --- which two investors may well disagree about --- drops out of the price entirely. That is the whole point of risk-neutral valuation.
